# 1. Context

This Notebook evaluates Character Error Rate (CER) & Word Error Rate (WER) in OCRed Document at paragraph compared to Ground Truth

> Accuracy evaluation will be scoped to document (1 page) and multi column layput in later studies

# 2. Imports

In [1]:
import jiwer
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1
import os
import json

In [2]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

In [3]:
from pathlib import Path

In [4]:
PROJECT_ID = os.getenv("PROJECT_ID", "")
API_LOCATION = os.getenv("API_LOCATION", "")

In [5]:
PROJECT_ID

'chrome-energy-464405-b6'

In [6]:
indic_parser_name = "indic_test_processor"

In [7]:
opts = ClientOptions(api_endpoint=f"{API_LOCATION}-documentai.googleapis.com")
client = documentai_v1.DocumentProcessorServiceClient(client_options=opts)
full_processor_name = client.processor_path(PROJECT_ID, API_LOCATION, "5945bfe7932ca5b7")
request = documentai_v1.GetProcessorRequest(name=full_processor_name)
processor = client.get_processor(request=request)

# Language: Hindi

In [8]:
class DocumentAIOCR:
    """A class to handle Google Document AI OCR operations"""
    
    def __init__(self, parser_nm: str, processor_id: str = "5945bfe7932ca5b7"):
        """
        Initialize Google Document AI Processor
        
        Args:
            parser_nm: Name of the parser
            processor_id: ID of the Document AI processor
        """
        self.api_location = os.getenv("API_LOCATION", "")
        self.project_id = os.getenv("PROJECT_ID", "")
        
        if not self.api_location or not self.project_id:
            raise ValueError("API_LOCATION and PROJECT_ID environment variables must be set")
            
        self.client = self._initialize_client()
        self.processor = self._initialize_processor(processor_id)
    
    def _initialize_client(self) -> documentai_v1.DocumentProcessorServiceClient:
        """Initialize and return Document AI client"""
        client_options = ClientOptions(
            api_endpoint=f"{self.api_location}-documentai.googleapis.com"
        )
        return documentai_v1.DocumentProcessorServiceClient(
            client_options=client_options
        )
    
    def _initialize_processor(self, processor_id: str) -> documentai_v1.Processor:
        """Initialize and return Document AI processor"""
        processor_name = self.client.processor_path(
            self.project_id, 
            self.api_location, 
            processor_id
        )
        request = documentai_v1.GetProcessorRequest(name=processor_name)
        return self.client.get_processor(request=request)
    
    def _read_image(self, img_path: str) -> bytes:
        """Read image file and return bytes content"""
        try:
            with open(img_path, "rb") as image:
                return image.read()
        except IOError as e:
            raise IOError(f"Error reading image file {img_path}: {str(e)}")
    
    def perform_ocr(self, img_path: str) -> documentai_v1.Document:
        """
        Perform OCR for a given image path
        
        Args:
            img_path: Path to the image file
            
        Returns:
            Document object containing OCR results
        """
        img_content = self._read_image(img_path)
        raw_document = documentai_v1.RawDocument(
            content=img_content,
            mime_type="image/png"
        )
        
        try:
            request = documentai_v1.ProcessRequest(
                name=self.processor.name,
                raw_document=raw_document
            )
            result = self.client.process_document(request=request)
            return result.document
        except Exception as e:
            raise RuntimeError(f"OCR processing failed: {str(e)}")
    
    async def perform_ocr_async(self, img_path: str) -> documentai_v1.Document:
        """Asynchronous version of perform_ocr"""
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.perform_ocr, img_path)
    
    async def process_multiple_images(self, img_paths: list) -> list:
        """
        Process multiple images concurrently
        
        Args:
            img_paths: List of image paths to process
            
        Returns:
            List of Document objects containing OCR results
        """
        tasks = [self.perform_ocr_async(img_path) for img_path in img_paths]
        return await asyncio.gather(*tasks, return_exceptions=True)

In [9]:
# Create DocumentAIOCR instance
document_ai_obj = DocumentAIOCR(parser_nm=indic_parser_name)

In [10]:
_PATH_SYNTH_DATA_ = Path("../../synth_data")
_LANG_ = "hindi"
_PATH_IMAGES_LANG_ = _PATH_SYNTH_DATA_.joinpath(_LANG_)

_PATH_IMAGES_LIST_PNG_ = list(_PATH_IMAGES_LANG_.glob("images/*/*.png"))
_PATH_IMAGES_LIST_GT_ = list(_PATH_IMAGES_LANG_.glob("gt/*.json"))

In [11]:
# Process multiple images concurrently
results = asyncio.run(document_ai_obj.process_multiple_images(_PATH_IMAGES_LIST_PNG_))

In [14]:
ocr_results_list_dict = []
for IDX, IMG_PATH in enumerate(_PATH_IMAGES_LIST_PNG_):
    degradation_level = IMG_PATH.parent.name
    gt_file_name = IMG_PATH.name.split("_n_pages_")[0]
    ocr_results_list_dict.append({"file_id": gt_file_name, "degradation_level": degradation_level.split("_")[0][0] + "_" +degradation_level.split("_")[1] , "ocr_output_raw": results[IDX].text})

In [16]:
ocr_results_list_dict

[{'file_id': 'Devanagari_Noto_Sans_13',
  'degradation_level': 'l_3',
  'ocr_output_raw': 'अरामेक मारिया ट्रक में में\nमोनिका एन्ना मारिया बेलुची (जन्म 30 सितंबर 1964) एक इतालवी अभिनेत्री और फ़ैशन मॉडल हैं।\nव्यक्तिगत जीवन.\nचित्रकार मारिया गुस्टीनेल्ली और एक ट्रक कंपनी के मालिक लुइगी बेलुची की बेटी, मोनिका बेलुची का जन्म सिट्टा डि कास्तेलो,\nउम्ब्रिया, इटली में हुआ था। बेलुची ने 16 वर्ष की उम्र में मॉडलिंग शुरू की, जब वह लीसो क्लैसिको में पढ़ रही थीं। शुरूआत में\nएक वकील के रूप में कॅरियर बनाते हुए, बेलुची ने पेरुगिया विश्वविद्यालय में अपने ट्यूशन के भुगतान के लिए मॉडलिंग शुरू की,\nलेकिन जीवन-शैली ने उन्हें क़ानून के अध्ययन से दूर कर दिया। वे इतालवी, फ्रांसीसी और अंग्रेजी धारा-प्रवाह बोलती हैं, स्पेनिश\nअर्द्ध-धाराप्रवाह और इन सभी भाषाओं में बोलते हुए अभिनय करने के अलावा, उन्होंने "द पैशन ऑफ़ द क्राइस्ट" में मैरी\nमैगडलीन की अपनी भूमिका के लिए अरामेक भाषा में भी अभिनय किया।\nबेलुची ने साथी अभिनेता विन्सेन्ट कैसेल से शादी की है, जिनके साथ वे कई फिल्मों में नज़र आई और उनकी देवा (12 सितं

In [17]:
import pandas as pd

In [18]:
df = pd.DataFrame(ocr_results_list_dict)

In [25]:
def get_ocr_results_df(image_paths, results):
    ocr_results_list_dict = []
    for IDX, IMG_PATH in enumerate(image_paths):
        degradation_level = IMG_PATH.parent.name
        gt_file_name = IMG_PATH.name.split("_n_pages_")[0]
        ocr_results_list_dict.append({"file_id": gt_file_name, "degradation_level": degradation_level.split("_")[0][0].upper() + "_" +degradation_level.split("_")[1] , "ocr_output_raw": results[IDX].text})

    df_result = pd.DataFrame(ocr_results_list_dict)

    # pivot#
    pivoted_df = df_result.pivot(index='file_id',columns='degradation_level', values='ocr_output_raw').reset_index()

    # Rename columns to include 'ocr_output_' prefix
    pivoted_df.columns.name = None  # Remove the columns name
    renamed_columns = {
        col: f'ocr_output_{col}' if col != 'file_id' else col 
        for col in pivoted_df.columns
    }
    pivoted_df = pivoted_df.rename(columns=renamed_columns)

    return pivoted_df

In [26]:
ocr_res_df = get_ocr_results_df(_PATH_IMAGES_LIST_PNG_, results)

In [27]:
ocr_res_df

,file_id,ocr_output_L_0,ocr_output_L_1,ocr_output_L_2,ocr_output_L_3
0,Devanagari_Anek_Devanagari_16,को और पक्ष सकता- चारों\nहिंदू धर्म में एकादशी ...,को और पक्ष सकता- चारों\nहिंदू धर्म में एकादशी ...,को और पक्ष सकता- चारों\nहिंदू धर्म में एकादशी ...,को और पक्ष सकता- चारों\nहिंदू धर्म में एकादशी ...
1,Devanagari_Anek_Devanagari_22,"सफेद-पीला, परिवर्तन पेस्ट"" देखा रक्त\nमवाद बैक...","सफेद-पीला, परिवर्तन पेस्ट"" देखा रक्त\nमवाद बैक...","सफेद-पीला, परिवर्तन पेस्ट"" देखा रक्त\nमवाद बैक...","सफेद-पीला, परिवर्तन पेस्ट"" देखा रक्त\nमवाद बैक..."
2,Devanagari_Anek_Devanagari_24,तारीख मक्का समय चार गए\n2009 सऊदी अरब के जेद्द...,तारीख मक्का समय चार गए\n2009 सऊदी अरब के जेद्द...,तारीख मक्का समय चार गए\n2009 सऊदी अरब के जेद्द...,तारीख मक्का समय चार गए\n2009 सऊदी अरब के जेद्द...
3,Devanagari_Anek_Devanagari_3,IUCN दें: आधार होने प्रजातियों\nलुप्तप्राय प्र...,IUCN दें: आधार होने प्रजातियों\nलुप्तप्राय प्र...,IUCN दें: आधार होने प्रजातियों\nलुप्तप्राय प्र...,IUCN दें: आधार होने प्रजातियों\nलुप्तप्राय प्र...
4,Devanagari_Anek_Devanagari_5,के है अधिकांश भी दूरभाष\nउपग्रह दूरभाष (अंग्रे...,के है अधिकांश भी दूरभाष\nउपग्रह दूरभाष (अंग्रे...,के है अधिकांश भी दूरभाष\nउपग्रह दूरभाष (अंग्रे...,के है अधिकांश भी दूरभाष\nउपग्रह दूरभाष (अंग्रे...
5,Devanagari_Noto_Sans_10,बन 1970 उत्पाद प्रदान सिस्टम\nलोकल एरिया नेटवर...,बन 1970 उत्पाद प्रदान सिस्टम\nलोकल एरिया नेटवर...,बन 1970 उत्पाद प्रदान सिस्टम\nलोकल एरिया नेटवर...,बन 1970 उत्पाद प्रदान सिस्टम\nलोकल एरिया नेटवर...
6,Devanagari_Noto_Sans_13,अरामेक मारिया ट्रक में में\nमोनिका एन्ना मारिय...,अरामेक मारिया ट्रक में में\nमोनिका एन्ना मारिय...,अरामेक मारिया ट्रक में में\nमोनिका एन्ना मारिय...,अरामेक मारिया ट्रक में में\nमोनिका एन्ना मारिय...
7,Devanagari_Noto_Sans_15,जो एडोब प्रमुख फ़ोटोशॉप जिसे\nअडोबी फोटोशॉप (य...,जो एडोब प्रमुख फ़ोटोशॉप जिसे\nअडोबी फोटोशॉप (य...,जो एडोब प्रमुख फ़ोटोशॉप जिसे\nअडोबी फोटोशॉप (य...,जो एडोब प्रमुख फ़ोटोशॉप जिसे\nअडोबी फोटोशॉप (य...
8,Devanagari_Noto_Sans_18,और शताब्दियों (Y) तथा है।\nपृथ्वी का चुंबकीय क...,और शताब्दियों (Y) तथा है।\nपृथ्वी का चुंबकीय क...,और शताब्दियों (Y) तथा है।\nपृथ्वी का चुंबकीय क...,और शताब्दियों (Y) तथा है।\nपृथ्वी का चुंबकीय क...
9,Devanagari_Noto_Sans_2,ई सीना पेशेवर रहे। प्रतियोगिता\nजॉन फेलिक्स एं...,ई सीना पेशेवर रहे। प्रतियोगिता\nजॉन फेलिक्स एं...,ई सीना पेशेवर रहे। प्रतियोगिता\nजॉन फेलिक्स एं...,ई सीना पेशेवर रहे। प्रतियोगिता\nजॉन फेलिक्स एं...


In [ ]:
def read_json(path) -> dict:
    """
    Reads JSON and properly decodes Indic text
    
    Args:
        path: Path to JSON file
    Returns:
        dict: Decoded JSON data with proper Indic text rendering
    """
    with open(path, 'r', encoding='utf-8') as file:
        data = json.load(file)
        
        # Handle the text fields with proper Unicode handling
        if 'header' in data:
            data['header'] = data['header'].encode('utf-8').decode('utf-8')
        if 'full_text' in data:
            data['full_text'] = data['full_text'].encode('utf-8').decode('utf-8')
            
        return data

In [ ]:
file_id_gt_dict = []
for file_gt in _PATH_IMAGES_LIST_GT_:
    file_nm = file_gt.name.split(".")[0]
    gt_json = read_json(file_gt)
    file_id_gt_dict.append({"file_id": file_nm, "ground_truth": (gt_json['header'] + "\n" + gt_json['full_text']).replace("\n", " ")})

In [ ]:
gt_df = pd.DataFrame(file_id_gt_dict)

In [ ]:
gt_df

In [ ]:
gt_df_ocr = pd.merge(gt_df, ocr_res_df, left_on='file_id', right_on='file_id')

In [ ]:
gt = gt_df_ocr.iloc[7]['ground_truth']
ocr = gt_df_ocr.iloc[7]['ocr_output_level_0']

In [ ]:
gt_flat = gt.replace("\n", " ")
ocr_flat = ocr.replace("\n", " ")

In [ ]:
output = jiwer.process_characters("तीन मील चन्‍द्रावली स्‍थापना", "तीन मील चन्द्रावली स्थापना")

In [ ]:
print(jiwer.visualize_alignment(output))

In [ ]:
print(jiwer.visualize_error_counts(output))

In [ ]:
output_wer = jiwer.process_words("तीन मील चन्‍द्रावली स्‍थापना", "तीन मील चन्द्रावली स्थापना")

In [ ]:
print(jiwer.visualize_alignment(output_wer))

In [ ]:
print(jiwer.visualize_error_counts(output_wer))

In [28]:
zip([1,2,3], [4,5,6])